# 09.3 — Config composition

Every cluster job ([09.2](09.2_slurm_dispatch.ipynb)) starts by building its configuration. Rather than one giant config file per experiment (repeating dozens of shared settings), the pipeline **composes** a config: a `base.yaml` of defaults, overlaid with a per-experiment `target_milestone/<name>.yaml`, then CLI overrides. This is how one base spec serves every experiment — you write only what *differs*. A note up front, and it's this module's recurring lesson: the notebook is filed under "hydra_config_composition," and `hydra-core` is a declared dependency, but **the actual composition is plain OmegaConf `merge` — there is no Hydra here.** Read the code, not the label.

This notebook covers:

* Why layered config (base + overrides) beats one file per experiment.
* `OmegaConf.merge` — how `base.yaml` + a target compose (target wins).
* The honest state: OmegaConf, not Hydra, despite the name.
* CLI override precedence and the real preset files.

**Prerequisites:** [00.7 pip & packaging](../00_orientation/00.7_pip_packaging_and_project_anatomy.ipynb), [09.2 SLURM dispatch](09.2_slurm_dispatch.ipynb).

## Section 1 — What MATLAB does

MATLAB configures runs through a *cascade* of parameter functions — `PARAMETERS_cgg_runAutoEncoder.m` sets defaults, and `PARAMETERS_OPTIMAL_cgg_runAutoEncoder_v3.m` (and friends) override them for specific experiments:

```matlab
cfg = PARAMETERS_cgg_runAutoEncoder();          % base defaults
cfg = PARAMETERS_OPTIMAL_cgg_runAutoEncoder_v3(cfg);   % overlay experiment overrides
% ... each PARAMETERS_* function returns a struct with some fields changed
```

It's a layered override pattern expressed as a chain of functions. The port replaces the function cascade with *YAML files* composed by OmegaConf: `base.yaml` mirrors `PARAMETERS_cgg_runAutoEncoder`, and each `target_milestone/<name>.yaml` is one experiment's override layer. Same idea — base plus overrides — but declarative.

## Section 2 — The Python concepts you need

### 2.1 — Why layered config

The base config has ~70 fields. If every experiment were a standalone file, changing one shared default (say the gradient-clip threshold) would mean editing dozens of files, and they'd drift out of sync. Layering fixes this: `base.yaml` holds the shared defaults *once*; each experiment file lists *only its differences* (a handful of keys). Change a default in `base.yaml` and every experiment inherits it. This is the config version of "don't repeat yourself" — and it's why a target file is small even though the effective config is large.

### 2.2 — `OmegaConf.merge`: base + target

In [ ]:
from omegaconf import OmegaConf

base = OmegaConf.load("../../configs/base.yaml")
target = OmegaConf.load("../../configs/target_milestone/A_logistic_synthetic.yaml")
merged = OmegaConf.merge(base, target)          # target keys WIN over base

print(f"base has {len(base)} keys; target lists only {len(target)}; merged has {len(merged)}.")
print()
print("base   model_name:", base.model_name)
print("target model_name:", target.model_name)
print("merged model_name:", merged.model_name, "  ← the target's value won")

`OmegaConf.merge(base, target)` deep-merges the two: every key in `target` overrides `base`, and keys only in `base` carry through untouched. So the milestone-A target lists just its ~30 differences (it swaps the model to `Logistic Regression`, adjusts a few settings), and the merge fills in the other ~40 fields from `base`. The result is a complete config where the *experiment* only had to state what makes it different. This is exactly the MATLAB cascade (§1), done declaratively.

### 2.3 — The honest state: OmegaConf, not Hydra

In [ ]:
from pathlib import Path
cli = Path("../../src/neural_data_decoding/cli.py").read_text()
print("cli.py imports/uses Hydra?    ", "import hydra" in cli or "hydra.compose" in cli)
print("cli.py uses OmegaConf.merge?  ", "OmegaConf.merge" in cli)
print()
print("→ The composition is plain OmegaConf.merge. 'Hydra' appears in the notebook name and")
print("  as a declared dependency, but no Hydra compose()/initialize() is used. Read the code.")

[Hydra](https://hydra.cc) is a config framework built *on top of* OmegaConf that adds config *groups*, a `defaults:` composition list, command-line override syntax, and multirun. This project declares `hydra-core` as a dependency and even names this notebook "hydra_config_composition" — but the actual code uses only OmegaConf's `merge`. The `defaults:` blocks you'll see in the YAML files are inert (dropped before use); `configs/architecture/` and `configs/sweep/` are *empty* directories (architecture is chosen by code registries, [09.6](09.6_extending_the_pipeline.ipynb), not a Hydra config group).

This is the fourth "read the code, not the label" moment in the curriculum ([06.7](../06_loss_orchestration/06.7_the_confidence_pd_controller.ipynb) the "PD" controller, [07.5](../07_dynamic_curriculum/07.5_freeze_unfreeze_curriculum.ipynb) "requires_grad" freezing, [08.5](../08_output_and_analysis/08.5_weights_and_biases_integration.ipynb) the unwired W&B, and now "Hydra"). The lesson isn't that the naming is *bad* — Hydra may be where it's headed — but that you verify what the code *does*, not what a filename or dependency *suggests*. A simple `OmegaConf.merge` is honestly all this pipeline needs today.

### 2.4 — CLI override precedence

In [ ]:
# On top of the merged config, CLI flags apply in a defined order (later wins):
from omegaconf import OmegaConf
cfg = OmegaConf.merge(base, target)
print("after merge:            initial_learning_rate =", cfg.initial_learning_rate)

# A --override KEY=VALUE flag (applied via OmegaConf.update):
OmegaConf.update(cfg, "initial_learning_rate", 5e-3)
print("after --override:       initial_learning_rate =", cfg.initial_learning_rate)
print()
print("Full precedence (later wins): base → target → --sweep-index → --session → --override → --fold")

The effective config is built in layers, each overriding the last:

1. **`base.yaml`** — the defaults.
2. **`target_milestone/<name>.yaml`** — the experiment (via `--config-name`).
3. **`--sweep-index`** — the sweep point's overrides ([09.2](09.2_slurm_dispatch.ipynb), [09.4](09.4_parameter_sweeps.ipynb)).
4. **`--session`** — which recording session.
5. **`--override KEY=VALUE`** — ad-hoc overrides (repeatable).
6. **`--fold`** — the final say on the fold.

Later layers win, so a `--override` beats the target which beats the base. This lets one base config plus a target plus a few flags produce *any* specific run — which is exactly what the array job needs, since every task differs only by `--sweep-index` and `--session-run-idx`.

### 2.5 — The real preset files

In [ ]:
from pathlib import Path
targets = sorted(p.name for p in Path("../../configs/target_milestone").glob("*.yaml"))
print("target_milestone presets (the --config-name choices):")
for t in targets:
    print("  ", t)
print()
# The empty groups (a Hydra tell that isn't used):
for group in ("architecture", "sweep"):
    files = list(Path(f"../../configs/{group}").glob("*.yaml"))
    print(f"configs/{group}/ has {len(files)} yaml files (empty → not a config group)")

Five `target_milestone` presets ship — the milestone experiments (`A_logistic`, `B_gru_classifier`, `C_optimal`, `C_two_stage`) plus `real_data_base`. Each is a `--config-name` you can pass to `train` or `sweep-emit-slurm`. The `architecture/` and `sweep/` directories exist (a vestige of the Hydra-group idea) but are *empty* — confirming §2.3: architecture selection is code-driven ([09.6](09.6_extending_the_pipeline.ipynb)), and the sweep is the hand-authored dispatcher table ([09.4](09.4_parameter_sweeps.ipynb)), not a YAML config group.

## Section 3 — The `neural_data_decoding` implementation

`_load_config` — the two-file OmegaConf merge:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/cli.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "def _load_config" in line)
for k in range(i, i + 14):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `_load_config(name)` loads `base.yaml` and `target_milestone/<name>.yaml`, then `OmegaConf.merge(base, target)` — that one `merge` call is the whole composition (§2.2). No Hydra `compose`.
* Both files are required — a missing base or target raises `FileNotFoundError` with the path, so a typo'd `--config-name` fails loudly, not silently.
* The `assert isinstance(merged, DictConfig)` narrows the type (OmegaConf can return a list config); the return is always a dict-style config.
* CLI overrides are applied *after* this by `_apply_cfg_flags` (sweep index, session, `--override`, fold) via `OmegaConf.update` — the §2.4 precedence.
* `CONFIG_ROOT = <repo>/configs`; the composed config is written back out as `EncodingParameters.yaml` ([08.1](../08_output_and_analysis/08.1_folder_hierarchy_generation.ipynb)) with a stable schema (Critical Note #25) so every run records its full effective config.

## Section 4 — Hands-on exercises

### Exercise 4.1 — only the differences

Show that the target file lists far fewer keys than the merged result — quantifying how much the base layer contributes.

In [ ]:
# Reveal:
base = OmegaConf.load("../../configs/base.yaml")
for name in ("A_logistic_synthetic", "C_optimal_synthetic"):
    t = OmegaConf.load(f"../../configs/target_milestone/{name}.yaml")
    m = OmegaConf.merge(base, t)
    print(f"{name}: target lists {len(t)} keys → merged has {len(m)} "
          f"({len(m) - len(t)}+ inherited from base)")
print("→ each experiment states only its differences; base fills in the rest.")

### Exercise 4.2 — override beats target

Take a value set by the target, override it, and confirm the override wins — demonstrating the §2.4 precedence.

In [ ]:
# Reveal:
t = OmegaConf.load("../../configs/target_milestone/A_logistic_synthetic.yaml")
m = OmegaConf.merge(base, t)
target_model = m.model_name
OmegaConf.update(m, "model_name", "GRU")     # a --override model_name=GRU
print(f"target set model_name={target_model!r}; after --override → {m.model_name!r}")
print("→ later layers win: --override beats the target, which beats the base.")

## Section 5 — Common errors

### Looking for Hydra composition

There isn't any (§2.3) — it's `OmegaConf.merge`. Don't reach for `@hydra.main`, `hydra.compose`, or config-group syntax; they aren't wired. Compose with `OmegaConf.merge` and override with `OmegaConf.update`.

### A setting from the target isn't taking effect

Merge order matters — `merge(base, target)` lets the *target* win. If you accidentally `merge(target, base)`, base wins and the experiment's overrides are lost. Base first, target second.

### `FileNotFoundError` on `--config-name`

The name must match a file in `configs/target_milestone/` (§2.5), minus the `.yaml`. A typo (or expecting an `architecture/`/`sweep/` group) fails — those groups are empty.

### An override silently does nothing

`--override KEY=VALUE` uses the exact config key. A misspelled key adds a *new* key instead of overriding, so the intended field keeps its old value. Check the key exists in the merged config.

### Editing `base.yaml` to change one experiment

That changes *every* experiment (they all inherit base, §2.1). Put experiment-specific changes in the `target_milestone/` file or a `--override`, not in `base.yaml`.

## Section 6 — Further reading

- [`src/neural_data_decoding/cli.py`](../../src/neural_data_decoding/cli.py) — `_load_config`, `_apply_cfg_flags`.
- [OmegaConf docs](https://omegaconf.readthedocs.io/) — `merge`, `update`, and the config API this actually uses.
- [Hydra docs](https://hydra.cc) — what the *name* refers to, for when/if the project adopts it.

Next up: **[09.4 parameter sweeps](09.4_parameter_sweeps.ipynb)** — the 147-point sweep table and the 47-variable coverage matrix behind `--sweep-index`.